# Column with node-to-surface constraints — v3

Same column + `node_to_surface` constraints as v2, but with a different **base fixity strategy** so that `ops.nodeReaction(...)` works directly without having to sum across the full constraint chain.

## Why v3?

In v2, the base reference point (node 33) was fixed with `ops.fix(33, 1,1,1,1,1,1)`. That worked mechanically (displacements were correct), but `ops.nodeReaction(33)` returned **all zeros** — because node 33 has no element attached to it, and `nodeReaction` only reports the local element+load unbalance at a node. The restraining force was distributed across the `rigidLink`→`equalDOF` chain and had to be recovered by summing reactions at every node at `z=0`.

In v3 we instead:

1. Create a **grounded helper node** co-located with the base reference point, fixed in all 6 DOFs.
2. Connect it to the base reference point with a **stiff 6-DOF `zeroLength` element**.
3. Leave the base reference point itself *unfixed* — its restraint now comes through the zero-length spring.

Now `ops.nodeReaction(ground_tag)` returns the base reaction directly, and `ops.eleForce(zl_tag)` gives the same thing via the element's internal forces. Either works.

In [2]:
from apeGmsh import apeGmsh, Part
from apeGmsh import FEMData
from apeGmsh.sections import W_solid
import numpy as np
import pandas as pd
from pathlib import Path

In [3]:
m1 = apeGmsh(model_name='beam_column', verbose=False)
m1.begin()

column = m1.sections.W_solid(bf=150, tf=20, h=300, tw=10, length=2000, label='column')

m1.model.select(None, dim=3).to_physical(name='pg_column')

base = m1.model.geometry.add_point(x=0, y=0, z=0,    lc=100, label='base')
top  = m1.model.geometry.add_point(x=0, y=0, z=2000, lc=100, label='top')

m1.constraints.node_to_surface('base', column.labels.start_face)
m1.constraints.node_to_surface('top',  column.labels.end_face)

m1.loads.point.force(target='top', force=[0, 1000, 0])

m1.mesh.sizing.set_global_size(100)
m1.mesh.generation.generate(dim=3)

fem = m1.mesh.queries.get_fem_data(remove_orphans=True)

# Interactive windows; headless runs (nbconvert / CI) set APEGMSH_SKIP_VIEWER.
import os
if not os.environ.get('APEGMSH_SKIP_VIEWER'):
    m1.model.viewer()
    m1.mesh.viewer()

m1.end()

C:\Users\nmb\AppData\Local\Temp\ipykernel_22864\2799149036.py:1: UnitNotDeclaredWarning: apeGmsh(...): no consistent unit system declared (pass units='kN_m_s' or a baseUnits.systems module such as units=kN_m_s); Results and apeGmshViewer will require manual unit selection for unit conversions.
  m1 = apeGmsh(model_name='beam_column', verbose=False)


## Build and run the OpenSees model

DOF layout is identical to v2:

- Global `ndf = 6` — so reference + phantom nodes default to 6 DOF and accept 6-tuple `(Fx,Fy,Fz,Mx,My,Mz)` loads from `NodalLoadRecord`.
- Solid column nodes overridden to `ndf = 3`.
- `constraints('Penalty', 1e15, 1e15)` — `Transformation` can't be used because the constraint chain is daisy-chained (each phantom is both a `rigidLink` slave and an `equalDOF` master).

**New in v3: the zero-length base fixity.** We create a helper node at the same location as `base` with a fresh tag (`GROUND_TAG`), fix it in all 6 DOFs, and connect it to the base reference point with a 6-DOF stiff `zeroLength` element. The spring stiffness is chosen large enough that the extra compliance it introduces at the base is numerically negligible:

- Translational:  `θ_residual·K_rot = M_base` →  `θ = 2·10⁶ / K`. For `K = 10¹⁴`, `θ ≈ 2·10⁻⁸ rad`, which translates to `θ·L ≈ 4·10⁻⁵ mm` of extra tip motion — well below the mesh-discretisation noise.
- Translational:  `u = F / K = 10³ / 10¹⁴ = 10⁻¹¹ mm` — pure noise.

`K = 10¹⁴` stays 1 order of magnitude below the Penalty value (10¹⁵), so the system's condition number is still healthy for UmfPack.

In [4]:
from apeGmsh.opensees.emitter.live import get_ops
ops = get_ops()   # the OpenSees module apeGmsh drives

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

ops.timeSeries('Linear', 1)

# --- Nodes -----------------------------------------------------------------
for nid, xyz in fem.nodes.select(target='pg_column'):
    ops.node(nid, *xyz, '-ndf', 3)

ref = fem.nodes.select(target=['base', 'top'])
for nid, xyz in ref:
    ops.node(nid, *xyz)

for nid, xyz in fem.nodes.constraints.phantom_nodes():
    ops.node(nid, *xyz)

# --- Grounded helper node + zeroLength spring at the base reference point --
base_id = int(next(iter(fem.nodes.select(target='base').ids)))
base_xyz = next(xyz for nid, xyz in ref if nid == base_id)

GROUND_TAG = 10_000
ops.node(GROUND_TAG, *base_xyz)
ops.fix(GROUND_TAG, 1, 1, 1, 1, 1, 1)

K_SPRING = 1e14
for i in range(6):
    ops.uniaxialMaterial('Elastic', 100 + i, K_SPRING)

ZL_TAG = 20_000
ops.element('zeroLength', ZL_TAG, GROUND_TAG, base_id,
            '-mat', 100, 101, 102, 103, 104, 105,
            '-dir', 1, 2, 3, 4, 5, 6)

# Note: node 'base' is NOT ops.fix'd any more — restraint now comes through the zl spring.

# --- Material + tet elements ----------------------------------------------
E  = 200e3   # steel, N/mm^2
nu = 0.3
ops.nDMaterial('ElasticIsotropic', 1, E, nu)

for group in fem.elements.select(element_type='tet4').groups():
    for eid, conn in group:
        ops.element('FourNodeTetrahedron', int(eid), *[int(n) for n in conn], 1)

# --- MP constraints from apeGmsh -------------------------------------------
for master, slaves in fem.nodes.constraints.rigid_link_groups():
    for slave in slaves:
        ops.rigidLink('beam', int(master), int(slave))

for pair in fem.nodes.constraints.equal_dofs():
    ops.equalDOF(int(pair.master_node), int(pair.slave_node), *pair.dofs)

# --- Loads -----------------------------------------------------------------
ops.pattern('Plain', 1, 1)
for load in fem.nodes.loads.by_kind('nodal'):
    fx, fy, fz = load.force_xyz  or (0.0, 0.0, 0.0)
    mx, my, mz = load.moment_xyz or (0.0, 0.0, 0.0)
    ops.load(int(load.node_id), fx, fy, fz, mx, my, mz)

# --- Analysis --------------------------------------------------------------
ops.constraints('Penalty', 1e15, 1e15)
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-6, 10)
ops.algorithm('Linear')
ops.integrator('LoadControl', 0.1)
ops.analysis('Static')

for i in range(10):
    ok = ops.analyze(1)
    print(f'Step {i+1}: {"ok" if ok == 0 else f"failed ({ok})"}')

Step 1: ok
Step 2: ok
Step 3: ok
Step 4: ok
Step 5: ok
Step 6: ok
Step 7: ok
Step 8: ok
Step 9: ok


Step 10: ok


## Verify base reaction via `nodeReaction(ground)`

Because the zeroLength element is the only thing touching the ground node (and the reference point), `nodeReaction` has a concrete `K·u` to report. The forces on the two ends are equal and opposite by Newton's 3rd law, so:

- `nodeReaction(GROUND_TAG)` ≈ `-nodeReaction(base_id)` ≈ expected base reaction.
- `eleForce(ZL_TAG)` returns a 12-tuple `[F_ground (6), F_base (6)]` — same information, different API.

Expected (from equilibrium, `F_y = 1000` applied at `z = 2000`):

- `Fy_base = -1000 N`
- `Mx_base = +2·10⁶ N·mm`

In [5]:
ops.reactions()

rxn_ground = ops.nodeReaction(GROUND_TAG)
rxn_base   = ops.nodeReaction(base_id)
ele_force  = ops.eleForce(ZL_TAG)

print(f'nodeReaction(ground = {GROUND_TAG}):')
print(f'  Fx = {rxn_ground[0]:+.4e}   Fy = {rxn_ground[1]:+.4e}   Fz = {rxn_ground[2]:+.4e}')
print(f'  Mx = {rxn_ground[3]:+.4e}   My = {rxn_ground[4]:+.4e}   Mz = {rxn_ground[5]:+.4e}')

print(f'\nnodeReaction(base = {base_id}):   (should be equal and opposite)')
print(f'  Fx = {rxn_base[0]:+.4e}   Fy = {rxn_base[1]:+.4e}   Fz = {rxn_base[2]:+.4e}')
print(f'  Mx = {rxn_base[3]:+.4e}   My = {rxn_base[4]:+.4e}   Mz = {rxn_base[5]:+.4e}')

print(f'\neleForce(zl = {ZL_TAG}):   [F_ground(6) | F_base(6)]')
print(f'  ground end: {[f"{v:+.3e}" for v in ele_force[:6]]}')
print(f'  base end:   {[f"{v:+.3e}" for v in ele_force[6:]]}')

print(f'\nExpected (equilibrium):  Fy = -1.0000e+03   Mx = +2.0000e+06')
print(f'Error:                    Fy = {rxn_ground[1] + 1000.0:+.3e}   Mx = {rxn_ground[3] - 2.0e6:+.3e}')

nodeReaction(ground = 10000):
  Fx = -2.8201e-05   Fy = -1.0000e+03   Fz = +4.4290e-04
  Mx = +2.0000e+06   My = -1.8436e-01   Mz = -2.5340e+00

nodeReaction(base = 33):   (should be equal and opposite)
  Fx = +2.8201e-05   Fy = +1.0000e+03   Fz = -4.4290e-04
  Mx = -2.0000e+06   My = +1.8436e-01   Mz = +2.5340e+00

eleForce(zl = 20000):   [F_ground(6) | F_base(6)]
  ground end: ['-2.820e-05', '-1.000e+03', '+4.429e-04', '+2.000e+06', '-1.844e-01', '-2.534e+00']
  base end:   ['+2.820e-05', '+1.000e+03', '-4.429e-04', '-2.000e+06', '+1.844e-01', '+2.534e+00']

Expected (equilibrium):  Fy = -1.0000e+03   Mx = +2.0000e+06
Error:                    Fy = -1.235e-02   Mx = +2.748e+01


## Verify tip displacement against cantilever closed-form

Same check as v2. The W-section (`bf=150, tf=20, h=300, tw=10`) bends about its **strong (x) axis** under `Fy`:

`Ix = bf·H³/12 − (bf − tw)·h³/12`  where `H = h + 2·tf = 340 mm`

- `δ_EB  = P·L³ / (3·E·Ix)`  (Euler-Bernoulli, pure bending)
- `δ_T   = δ_EB + P·L / (G·As)`  (Timoshenko, adds shear compliance with `As ≈ tw·h`)

A tet4 solid mesh captures both modes, so the FEM tip displacement should land between the two bounds.

In [6]:
top_id = int(next(iter(fem.nodes.select(target='top').ids)))
disp = ops.nodeDisp(top_id)

bf, tf, h_web, tw, L = 150.0, 20.0, 300.0, 10.0, 2000.0
H  = 2 * tf + h_web
Ix = (bf * H**3) / 12 - ((bf - tw) * h_web**3) / 12
P  = 1000.0

delta_EB    = P * L**3 / (3 * E * Ix)
G           = E / (2 * (1 + nu))
As          = tw * h_web
delta_shear = P * L / (G * As)
delta_T     = delta_EB + delta_shear

print(f'Top node {top_id} displacement:')
print(f'  ux = {disp[0]:+.4e}   uy = {disp[1]:+.4e}   uz = {disp[2]:+.4e}')
print(f'\nSection properties:')
print(f'  Ix = {Ix:.4e} mm^4   As = {As:.1f} mm^2')
print(f'\nCantilever closed-form tip deflection:')
print(f'  delta_EB         = {delta_EB:.4e} mm   (bending only)')
print(f'  delta_shear      = {delta_shear:.4e} mm')
print(f'  delta_Timoshenko = {delta_T:.4e} mm')
print(f'  FEM uy           = {disp[1]:+.4e} mm')
print(f'\nRatios:')
print(f'  FEM / Euler-Bernoulli = {disp[1] / delta_EB:.4f}')
print(f'  FEM / Timoshenko      = {disp[1] / delta_T:.4f}')

Top node 34 displacement:
  ux = -5.7285e-05   uy = +7.8695e-02   uz = +2.4509e-06

Section properties:
  Ix = 1.7630e+08 mm^4   As = 3000.0 mm^2

Cantilever closed-form tip deflection:
  delta_EB         = 7.5629e-02 mm   (bending only)
  delta_shear      = 8.6667e-03 mm
  delta_Timoshenko = 8.4295e-02 mm
  FEM uy           = +7.8695e-02 mm

Ratios:
  FEM / Euler-Bernoulli = 1.0405
  FEM / Timoshenko      = 0.9336


## Record results

The cell below re-runs the analysis and records it, choosing the route by
the OpenSees build the deck ran on:

- **Ladruno fork:** ``ResolvedRecorderSpec.emit_default`` records the run
  with the fork's own recorder (a ``.ladruno`` file).
- **Stock openseespy:** the cell samples the live domain after every
  converged step and writes a native results file with ``NativeWriter``.
  That file holds **displacements only**: reaction forces (and, where a
  notebook records them, line stations) are recorded on the fork only.
  ``emit_default`` cannot record this deck on a stock build without MPCO:
  its ``.out`` route needs a bridge model archive that declares ``ndm``,
  and this deck was built by hand.

The fork's file is read back with ``Results.from_ladruno``, the native one
with ``Results.from_fem``. The deck tags nodes and elements by their FEM
ids, so the FEMData snapshot is the model. The cell then checks the
recorded last step against the live domain. ``Results.viewer`` honours ``APEGMSH_SKIP_VIEWER``, so this
notebook is safe to run under nbconvert / CI without changes.


In [7]:
# --- EOS-WIRING-V2 ---
# Record the run: the fork's own recorder on the Ladruno fork, a native
# file of displacements sampled from the live domain on stock openseespy.
import contextlib
import numpy as np
from apeGmsh import Results, workdir
from apeGmsh.opensees.emitter.live import get_backend_info, get_ops
from apeGmsh.results.spec import ResolvedRecorderRecord, ResolvedRecorderSpec
from apeGmsh.results.writers import NativeWriter

OUT = workdir()

# --- copied from cell 4 ---
from apeGmsh.opensees.emitter.live import get_ops
ops = get_ops()   # the OpenSees module apeGmsh drives

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

ops.timeSeries('Linear', 1)

# --- Nodes -----------------------------------------------------------------
for nid, xyz in fem.nodes.select(target='pg_column'):
    ops.node(nid, *xyz, '-ndf', 3)

ref = fem.nodes.select(target=['base', 'top'])
for nid, xyz in ref:
    ops.node(nid, *xyz)

for nid, xyz in fem.nodes.constraints.phantom_nodes():
    ops.node(nid, *xyz)

# --- Grounded helper node + zeroLength spring at the base reference point --
base_id = int(next(iter(fem.nodes.select(target='base').ids)))
base_xyz = next(xyz for nid, xyz in ref if nid == base_id)

GROUND_TAG = 10_000
ops.node(GROUND_TAG, *base_xyz)
ops.fix(GROUND_TAG, 1, 1, 1, 1, 1, 1)

K_SPRING = 1e14
for i in range(6):
    ops.uniaxialMaterial('Elastic', 100 + i, K_SPRING)

ZL_TAG = 20_000
ops.element('zeroLength', ZL_TAG, GROUND_TAG, base_id,
            '-mat', 100, 101, 102, 103, 104, 105,
            '-dir', 1, 2, 3, 4, 5, 6)

# Note: node 'base' is NOT ops.fix'd any more — restraint now comes through the zl spring.

# --- Material + tet elements ----------------------------------------------
E  = 200e3   # steel, N/mm^2
nu = 0.3
ops.nDMaterial('ElasticIsotropic', 1, E, nu)

for group in fem.elements.select(element_type='tet4').groups():
    for eid, conn in group:
        ops.element('FourNodeTetrahedron', int(eid), *[int(n) for n in conn], 1)

# --- MP constraints from apeGmsh -------------------------------------------
for master, slaves in fem.nodes.constraints.rigid_link_groups():
    for slave in slaves:
        ops.rigidLink('beam', int(master), int(slave))

for pair in fem.nodes.constraints.equal_dofs():
    ops.equalDOF(int(pair.master_node), int(pair.slave_node), *pair.dofs)

# --- Loads -----------------------------------------------------------------
ops.pattern('Plain', 1, 1)
for load in fem.nodes.loads.by_kind('nodal'):
    fx, fy, fz = load.force_xyz  or (0.0, 0.0, 0.0)
    mx, my, mz = load.moment_xyz or (0.0, 0.0, 0.0)
    ops.load(int(load.node_id), fx, fy, fz, mx, my, mz)

# --- Analysis --------------------------------------------------------------
ops.constraints('Penalty', 1e15, 1e15)
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-6, 10)
ops.algorithm('Linear')
ops.integrator('LoadControl', 0.1)
ops.analysis('Static')

# Record every FEM node the deck built.
_axes = ("x", "y", "z")[:3]
_built = set(int(n) for n in ops.getNodeTags())
_node_ids = np.asarray(
    [int(n) for n in fem.nodes.ids if int(n) in _built], dtype=np.int64)
# The deck drives the module get_ops() binds; get_backend_info() names it.
assert ops is get_ops(), "bind the deck with ops = get_ops()"
_backend = get_backend_info()
_fork = _backend.kind == "fork"
print(f"OpenSees build: {_backend.kind} ({_backend.source})")

if _fork:
    spec = ResolvedRecorderSpec(
        fem_snapshot_id=fem.snapshot_id,
        records=tuple(
            ResolvedRecorderRecord(
                category="nodes", name=kind,
                components=tuple(f"{kind}_{a}" for a in _axes),
                dt=None, n_steps=None, node_ids=_node_ids,
            )
            for kind in ("displacement", "reaction_force")
        ),
    )
    # emit_default needs a model archive on disk; the deck tags nodes and
    # elements by FEM id, so the FEMData archive is that model.
    model_h5 = OUT / "model.h5"
    fem.to_h5(str(model_h5))
    _recording = spec.emit_default(OUT / "run", model_h5=model_h5, ops=ops,
                                   overwrite=True)
else:
    _recording = contextlib.nullcontext()

_steps = []   # stock: (time, displacement) after every step


def _record_step():
    if not _fork:
        _steps.append((ops.getTime(), np.array(
            [ops.nodeDisp(int(n))[:3] for n in _node_ids])))


with _recording as run:
    for i in range(10):
        ok = ops.analyze(1)
        if ok == 0:
            _record_step()
        print(f'Step {i+1}: {"ok" if ok == 0 else f"failed ({ok})"}')

if _fork:
    results_path = run.path
    results = Results.from_ladruno(str(results_path), model_h5=str(model_h5))
else:
    results_path = OUT / "run_native.h5"
    if results_path.exists():
        results_path.unlink()
    _u = np.stack([u for _, u in _steps])          # (T, N, ndm)
    with NativeWriter(results_path) as _nw:
        _nw.open(fem=fem, source_type="native", opensees_backend=_backend.kind)
        _sid = _nw.begin_stage(name="analysis", kind="static",
                               time=np.array([t for t, _ in _steps]))
        _nw.write_nodes(_sid, "partition_0", node_ids=_node_ids, components={
            f"displacement_{a}": _u[:, :, k] for k, a in enumerate(_axes)})
        _nw.end_stage()
    results = Results.from_fem(fem, results_path, kind="native", cache_root=OUT)
print(f"recorded -> {results_path}")

# The recorded last step is the live domain's displacement field.
_slabs = [results.nodes.get(ids=_node_ids, component=f"displacement_{a}")
          for a in _axes]
assert sorted(int(n) for n in _slabs[0].node_ids) == sorted(_node_ids.tolist())
_rec = np.column_stack([s.values[-1] for s in _slabs])
_live = np.array([ops.nodeDisp(int(n))[:3] for n in _slabs[0].node_ids])
_scale = max(float(np.abs(_live).max()), 1e-300)
assert np.allclose(_rec, _live, rtol=1e-6, atol=1e-9 * _scale), (
    "the recorded displacement disagrees with the live domain")
print(f"recorded last step matches the live domain at {len(_node_ids)} nodes")


OpenSees build: fork (C:\Program Files\Ladruno\OpenSees\bin\opensees.pyd)
Step 1: ok
Step 2: ok
Step 3: ok


Step 4: ok


Step 5: ok


Step 6: ok
Step 7: ok
Step 8: ok
Step 9: ok
Step 10: ok


recorded -> outputs\run.ladruno
recorded last step matches the live domain at 642 nodes


In [8]:
# Open the recorded run in the apeGmsh ResultsViewer (subprocess, non-blocking).
# Set APEGMSH_SKIP_VIEWER=1 in the environment to skip the GUI under nbconvert / CI.
results.viewer(blocking=False)


[skip viewer] APEGMSH_SKIP_VIEWER set
